# 任务 3：亿级数据会话划分

## 任务描述
基于用户行为时间序列，使用窗口函数划分会话（Session）

## 逻辑定义
如果同一个用户连续两个行为的时间间隔（TimeDiff）超过了 30 分钟（1800 秒），
我们认为前一个会话已结束，后一个行为属于新会话。

## 实现步骤
1. 按照 `user_id` 分组并按 `timestamp` 排序
2. 计算相邻行为的时间差 `timediff`
3. 识别新会话起点（`timediff > 1800`），并使用累加和（`cum_sum`）生成全局唯一的 `session_id`

In [ ]:
import polars as pl
import time

print(f"Polars 版本：{pl.__version__}")

In [ ]:
# 配置
TIMEOUT_THRESHOLD = 1800  # 30 分钟 = 1800 秒
INPUT_PATH = "clean_data_deduped.parquet"  # 使用已去重的数据

# 性能计时开始
start_time = time.time()

## 1. 加载数据（Lazy 模式）

In [ ]:
# 加载数据
q = pl.scan_parquet(INPUT_PATH)
print(f"数据加载完成（LazyFrame 模式）")

## 2. 会话划分核心逻辑

### 使用的窗口函数
| 函数 | 作用 |
|------|------|
| `.shift(1).over("user_id", order_by="timestamp")` | 获取同用户上一行的时间戳 |
| `.cum_sum().over("user_id")` | 按用户累加新会话标记，生成会话序号 |

In [ ]:
# 步骤 1: 计算上一个行为的 timestamp（窗口函数 shift + over）
q_step1 = q.with_columns(
    pl.col("timestamp").shift(1).over("user_id", order_by="timestamp").alias("prev_timestamp")
)

# 步骤 2: 计算时间差（秒）
q_step2 = q_step1.with_columns(
    pl.when(pl.col("prev_timestamp").is_not_null())
    .then(pl.col("timestamp") - pl.col("prev_timestamp"))
    .otherwise(pl.lit(0))  # 第一个行为的时间差设为 0
    .alias("timediff")
)

# 步骤 3: 识别新会话起点（timediff > 1800 或 第一个行为）
q_step3 = q_step2.with_columns(
    pl.when(
        (pl.col("timediff") > TIMEOUT_THRESHOLD) | (pl.col("prev_timestamp").is_null())
    )
    .then(1)
    .otherwise(0)
    .alias("is_new_session")
)

# 步骤 4: 使用累加和生成全局唯一的 session_id
q_with_session = q_step3.with_columns(
    (
        pl.col("user_id").cast(pl.String) + "_" + 
        pl.col("is_new_session").cum_sum().over("user_id").cast(pl.String)
    ).alias("session_id")
)

print("会话 ID 计算完成")

## 3. 收集结果

In [ ]:
# 收集结果（使用流式引擎降低内存压力）
df_result = q_with_session.collect(engine="streaming")

# 性能计时结束
end_time = time.time()
total_time = end_time - start_time

print(f"结果收集完成")
print(f"总耗时：{total_time:.2f} 秒")

## 4. 结果统计

In [ ]:
print("=" * 60)
print("会话划分结果统计")
print("=" * 60)

# 基本统计
total_rows = df_result.height
unique_users = df_result["user_id"].n_unique()
unique_sessions = df_result["session_id"].n_unique()

print(f"总行为数：{total_rows:,}")
print(f"独立用户数：{unique_users:,}")
print(f"总会话数：{unique_sessions:,}")
print(f"平均每用户会话数：{unique_sessions / unique_users:.2f}")
print(f"平均每会话行为数：{total_rows / unique_sessions:.2f}")

## 5. 时间差统计

In [ ]:
print("=" * 60)
print("时间差（timediff）统计")
print("=" * 60)

timediff_stats = df_result["timediff"].describe()
print(timediff_stats)

## 6. 性能统计

In [ ]:
print("=" * 60)
print("性能统计")
print("=" * 60)

print(f"总耗时：{total_time:.2f} 秒")
print(f"处理速度：{total_rows / total_time:,.0f} 条/秒")
print(f"\n窗口函数性能感知：")
print(f"  - shift + over 计算：按 user_id 分组，按 timestamp 排序")
print(f"  - cum_sum + over 计算：与 shift 共享相同的窗口分组开销")
print(f"  - 总耗时包含：shift、timediff、cum_sum、collect 全部操作")

## 7. 保存结果

In [ ]:
output_path = "data_with_session.parquet"
df_result.write_parquet(output_path)
print(f"结果已保存至：{output_path}")

## 8. 数据预览

In [ ]:
print("=" * 60)
print("数据预览（前 10 条）")
print("=" * 60)

df_result.select([
    "user_id", 
    "timestamp", 
    "prev_timestamp", 
    "timediff", 
    "is_new_session", 
    "session_id"
]).head(10)

## 9. 新会话起点验证（查看 timediff > 1800 的样本）

In [ ]:
print("=" * 60)
print("新会话起点样本（timediff > 1800 秒）")
print("=" * 60)

df_result.filter(pl.col("timediff") > 1800).select([
    "user_id",
    "timestamp",
    "prev_timestamp",
    "timediff",
    "session_id"
]).head(10)